In [1]:
!pip install ultralytics==8.1.47 -q
!pip install albumentations -q
!pip install roboflow -q  # optional, không cần nếu đã upload zip

In [2]:
import zipfile, os, shutil
from pathlib import Path

# Kiểm tra
for split in ["train", "val"]:
    n_img = len(list(Path(f"/kaggle/input/datasets/ha10thmay/dataset-yolo/dataset_yolo/images/{split}").glob("*")))
    n_lbl = len(list(Path(f"/kaggle/input/datasets/ha10thmay/dataset-yolo/dataset_yolo/labels/{split}").glob("*")))
    print(f"{split}: {n_img} images, {n_lbl} labels")

# In thử 1 file label để kiểm tra
lbl_files = list(Path("/kaggle/input/datasets/ha10thmay/dataset-yolo/dataset_yolo/labels/train").glob("*.txt"))
print(f"\nSample label ({lbl_files[0].name}):")
print(open(lbl_files[0]).read())

train: 46 images, 46 labels
val: 11 images, 11 labels

Sample label (19_jpg.rf.bj8WIzzpgiUNISwFx2QS.txt):
1 0.196275 0.499510 0.065883 0.097059
1 0.484371 0.508927 0.075408 0.256609
1 0.631600 0.462866 0.046533 0.148339
1 0.804879 0.491228 0.064758 0.087416
1 0.484988 0.241517 0.063308 0.030900
0 0.167858 0.685179 0.252383 0.076240
2 0.657092 0.041742 0.649183 0.035040
2 0.655338 0.880807 0.649008 0.179146


In [3]:
import albumentations as A
import cv2, shutil, random
from pathlib import Path

def augment_split(img_dir, lbl_dir, out_img_dir, out_lbl_dir, n_aug=7):
    Path(out_img_dir).mkdir(parents=True, exist_ok=True)
    Path(out_lbl_dir).mkdir(parents=True, exist_ok=True)
    
    transform = A.Compose([
        A.RandomBrightnessContrast(
            brightness_limit=0.4, 
            contrast_limit=0.4, p=0.8),
        A.GaussNoise(std_range=(0.1, 0.3), p=0.5),
        A.Blur(blur_limit=3, p=0.3),
        A.Sharpen(p=0.3),
        A.CLAHE(clip_limit=4.0, p=0.5),        # rất tốt cho ảnh scan
        A.RandomGamma(p=0.4),
        A.HorizontalFlip(p=0.3),
        A.Rotate(limit=3, border_mode=cv2.BORDER_CONSTANT, p=0.3),
        A.RandomScale(scale_limit=0.2, p=0.4),
        A.Perspective(scale=(0.02, 0.05), p=0.3),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(10, 20),
            hole_width_range=(10, 20),
            p=0.2
        ),
    ], bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['labels'],
        min_visibility=0.4,
        clip=True
    ))
    
    img_files = list(Path(img_dir).glob("*.jpg")) + \
                list(Path(img_dir).glob("*.png")) + \
                list(Path(img_dir).glob("*.jpeg"))
    
    copied, augmented = 0, 0
    for img_path in img_files:
        lbl_path = Path(lbl_dir) / (img_path.stem + ".txt")
        
        # Luôn copy ảnh gốc
        shutil.copy(img_path, out_img_dir)
        shutil.copy(lbl_path, out_lbl_dir)
        copied += 1
        
        if not lbl_path.exists():
            continue
        
        # Đọc labels
        bboxes, labels = [], []
        with open(lbl_path) as f:
            for line in f.read().strip().split('\n'):
                if not line: continue
                parts = line.split()
                labels.append(int(parts[0]))
                bboxes.append([float(x) for x in parts[1:5]])
        
        if not bboxes:
            continue
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        # Augment n_aug lần
        for i in range(n_aug):
            try:
                result = transform(image=img, bboxes=bboxes, labels=labels)
                if not result['bboxes']:
                    continue
                
                out_name = f"{img_path.stem}_aug{i:02d}"
                
                cv2.imwrite(
                    f"{out_img_dir}/{out_name}.jpg",
                    result['image'],
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
                
                with open(f"{out_lbl_dir}/{out_name}.txt", 'w') as f:
                    for lbl, bbox in zip(result['labels'], result['bboxes']):
                        f.write(f"{lbl} "
                                f"{bbox[0]:.6f} {bbox[1]:.6f} "
                                f"{bbox[2]:.6f} {bbox[3]:.6f}\n")
                augmented += 1
            except Exception as e:
                pass
    
    total = len(list(Path(out_img_dir).glob("*.jpg")))
    print(f"  {split}: {copied} gốc + {augmented} aug = {total} tổng")

print("Đang augment...")
for split in ["train", "val"]:
    # Val chỉ augment nhẹ (3 lần) để tăng coverage
    n = 7 if split == "train" else 2
    augment_split(
        img_dir=f"/kaggle/input/datasets/ha10thmay/dataset-yolo/dataset_yolo/images/{split}",
        lbl_dir=f"/kaggle/input/datasets/ha10thmay/dataset-yolo/dataset_yolo/labels/{split}",
        out_img_dir=f"/kaggle/working/dataset_aug/images/{split}",
        out_lbl_dir=f"/kaggle/working/dataset_aug/labels/{split}",
        n_aug=n
    )
print("Done!")

Đang augment...


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


  train: 43 gốc + 301 aug = 333 tổng
  val: 11 gốc + 22 aug = 32 tổng
Done!


In [4]:
yaml_content = """path: /kaggle/working/dataset_aug
train: images/train
val: images/val
nc: 3
names: ['note', 'part-drawing', 'table']
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print(open("/kaggle/working/data.yaml").read())

path: /kaggle/working/dataset_aug
train: images/train
val: images/val
nc: 3
names: ['note', 'part-drawing', 'table']


In [5]:
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_SILENT"] = "true"

# Patch thêm: mock wandb trước khi ultralytics import nó
import sys, types
mock_wandb = types.ModuleType("wandb")
mock_wandb.run = None
mock_wandb.init = lambda **kwargs: None
mock_wandb.log = lambda *args, **kwargs: None
mock_wandb.finish = lambda: None
mock_wandb.Image = lambda *args, **kwargs: None
sys.modules["wandb"] = mock_wandb

import os, types, torch

# ── Fix 1: Tắt wandb ─────────────────────────────────────
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

# ── Fix 2: Patch ray TRƯỚC KHI import ultralytics ────────
try:
    import ray
    import ray.train
    import ray.train._internal
    # Tạo mock session module
    mock_session = types.ModuleType("ray.train._internal.session")
    mock_session._get_session = lambda: None
    import sys
    sys.modules["ray.train._internal.session"] = mock_session
    ray.train._internal.session = mock_session
except Exception as e:
    print(f"Ray patch note: {e}")

# ── Fix 3: Patch torch.load ──────────────────────────────
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

# ── Import sau khi đã patch ──────────────────────────────
from ultralytics import RTDETR
from ultralytics.utils import callbacks

# ── Fix 4: Remove raytune callback HOÀN TOÀN ────────────
# Cần làm sau khi import ultralytics
import ultralytics.utils.callbacks.raytune as raytune_cb
raytune_cb.on_fit_epoch_end = lambda trainer: None

# Override luôn trong registry
for event_name, cb_list in callbacks.default_callbacks.items():
    callbacks.default_callbacks[event_name] = [
        cb for cb in cb_list 
        if getattr(cb, '__module__', '') != 'ultralytics.utils.callbacks.raytune'
    ]

print("✓ All patches applied")

# ── Load model ───────────────────────────────────────────
model = RTDETR('rtdetr-l.pt')

# ── Fix 5: Remove khỏi model.callbacks sau khi khởi tạo ─
for event_name in list(model.callbacks.keys()):
    model.callbacks[event_name] = [
        cb for cb in model.callbacks[event_name]
        if getattr(cb, '__module__', '') != 'ultralytics.utils.callbacks.raytune'
    ]

print("✓ Callbacks cleaned:", {k: len(v) for k, v in model.callbacks.items() if v})

# ── TRAIN ────────────────────────────────────────────────
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=1024,
    batch=4,
    lr0=5e-5,
    lrf=0.01,
    warmup_epochs=5,
    weight_decay=1e-4,
    mosaic=0.5,
    copy_paste=0.3,
    patience=25,
    save=True,
    save_period=10,
    val=True,
    plots=True,
    project='runs',
    name='rtdetr_v1',
    device=0,
    workers=2,
    seed=42,
    verbose=True
)

print("\n=== FINAL METRICS ===")
print(f"mAP50:    {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

Ray patch note: No module named 'ray'
✓ All patches applied
✓ Callbacks cleaned: {'on_pretrain_routine_start': 1, 'on_pretrain_routine_end': 1, 'on_train_start': 1, 'on_train_epoch_start': 1, 'on_train_batch_start': 1, 'optimizer_step': 1, 'on_before_zero_grad': 1, 'on_train_batch_end': 1, 'on_train_epoch_end': 1, 'on_fit_epoch_end': 1, 'on_model_save': 1, 'on_train_end': 1, 'on_params_update': 1, 'teardown': 1, 'on_val_start': 1, 'on_val_batch_start': 1, 'on_val_batch_end': 1, 'on_val_end': 1, 'on_predict_start': 1, 'on_predict_batch_start': 1, 'on_predict_postprocess_end': 1, 'on_predict_batch_end': 1, 'on_predict_end': 1, 'on_export_start': 1, 'on_export_end': 1}
New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=rtdetr-l.pt, data=/kaggle/working/data.yaml, epochs=100, time=None, patience=25, batch=4,

2026-04-01 05:40:53.184948: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775022053.207362     385 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775022053.214603     385 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775022053.233610     385 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775022053.233628     385 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775022053.233631     385 computation_placer.cc:177] computation placer alr

Overriding model.yaml nc=80 with nc=3
WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1     25248  ultralytics.nn.modules.block.HGStem          [3, 32, 48]                   
  1                  -1  6    155072  ultralytics.nn.modules.block.HGBlock         [48, 48, 128, 3, 6]           
  2                  -1  1      1408  ultralytics.nn.modules.conv.DWConv           [128, 128, 3, 2, 1, False]    
  3                  -1  6    839296  ultralytics.nn.modules.block.HGBlock         [128, 96, 512, 3, 6]          
  4                  -1  1      5632  ultralytics.nn.modules.conv.DWConv           [512, 512, 3, 2, 1, False]    
  5                  -1  6   1695360  ultralytics.nn.modules.block.HGBlock         [512, 192, 1024, 5, 6, True, False]
  6                  -1  6   2055808  ultralytics.nn.modules.block.HGBlock         [1024, 192, 1024, 5, 

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/checks.py:640: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):


AMP: checks passed ✅


...

  0%|          | 0/86 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
    100/100      7.74G     0.1123     0.1913    0.05691         23       1024:  74%|███████▍  | 64/86 [00:43<00:15,  1.47it/s]libpng warning: iCCP: known incorrect sRGB profile
    100/100      7.74G     0.1095     0.1905    0.05593         30       1024:  85%|████████▍ | 73/86 [00:50<00:08,  1.46it/s]libpng warning: iCCP: known incorrect sRGB profile
    100/100      7.74G     0.1072     0.1882    0.05713         24    

                   all         33        235      0.942      0.905      0.935      0.829



100 epochs completed in 1.709 hours.
Optimizer stripped from runs/rtdetr_v19/weights/last.pt, 66.2MB
Optimizer stripped from runs/rtdetr_v19/weights/best.pt, 66.1MB

Validating runs/rtdetr_v19/weights/best.pt...
Ultralytics YOLOv8.1.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 498 layers, 31989905 parameters, 0 gradients, 103.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  2.16it/s]


                   all         33        235      0.926      0.929      0.942      0.843
                  note         33         33      0.995      0.909      0.906      0.718
          part-drawing         33        171      0.958      0.877      0.947      0.864
                 table         33         31      0.824          1      0.975      0.947
Speed: 0.6ms preprocess, 57.3ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs/rtdetr_v19

=== FINAL METRICS ===
mAP50:    0.9424
mAP50-95: 0.8431


In [6]:
# # Chạy cell này TRƯỚC, sau đó restart kernel, rồi chạy 
# !pip uninstall ray -y -q
# print("Ray uninstalled. Now restart kernel (Run > Restart & Clear Output) rồi chạy lại cell train.")

In [8]:
import zipfile
from pathlib import Path

# rtdetr_v19 là tên folder train của bạn
RUN_DIR = Path('/kaggle/working/runs/rtdetr_v19')

with zipfile.ZipFile('/kaggle/working/model_checkpoint.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(RUN_DIR / 'weights/best.pt', 'best.pt')
    z.write(RUN_DIR / 'weights/last.pt', 'last.pt')
    z.write('/kaggle/working/data.yaml',  'data.yaml')
    
    # Plots để viết báo cáo
    for f in RUN_DIR.glob('*.png'):
        z.write(f, f'plots/{f.name}')
    for f in RUN_DIR.glob('*.csv'):
        z.write(f, f'logs/{f.name}')

size = Path('/kaggle/working/model_checkpoint.zip').stat().st_size / 1e6
print(f"✓ model_checkpoint.zip ({size:.1f} MB)")
print("→ Vào tab Output bên phải → Download")

✓ model_checkpoint.zip (122.8 MB)
→ Vào tab Output bên phải → Download
